# 05 — Validate routing outputs

Validate source coverage, graph status, supported schemas, non-negative and nested accessibility contours, lagged-stock provenance, and firm self-exclusion against the long Fachgruppe table.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"D:\CO2_Masterarbeit\CO2_Masterarbeit")
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
POI_DIR = PROJECT_DIR / "TOOLS" / "pois"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
STATUS_PATH = ROUTING_DATA / "status" / "routing_feature_status.csv"
GRAPH_STATUS_PATH = ROUTING_DATA / "status" / "graph_build_status.csv"
REPORT_PATH = ROUTING_DATA / "reports" / "routing_validation_summary.csv"
YEARS = range(2015, 2026)
sys.path.insert(0, str(PROJECT_DIR / "TOOLS" / "routing"))
from routing_utils import ACCESS_MINUTES, fachgruppe_ids, main_access_columns
FACHGRUPPE_IDS = fachgruppe_ids(PANEL_PATH)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def result(year, product, check, passed):
    return {"year": year, "product": product, "check": check, "status": "OK" if bool(passed) else "CHECK"}

def numeric_checks(frame, columns, year, product):
    rows = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors="coerce") if column in frame else pd.Series(dtype=float)
        rows += [result(year, product, f"{column} numeric", len(values) == len(frame) and values.notna().all()), result(year, product, f"{column} non-negative", (values >= 0).all())]
    for prefix in ["pop_access", "existing_firms_access", "same_fachgruppe_firms_access"]:
        left, right = f"{prefix}_15min", f"{prefix}_30min"
        if {left, right}.issubset(frame): rows.append(result(year, product, f"{left} <= {right}", (frame[left] <= frame[right]).all()))
    return rows

records = []
for source_path in [PANEL_PATH, ACTIVE_CELLS_PATH, ANAL_DATA / "firms_assigned_100m.geoparquet"]:
    records.append(result("all", "source", f"{source_path.name} exists", source_path.exists()))
active_count = len(pd.read_parquet(ACTIVE_CELLS_PATH, columns=["grid_id"]))
for year in YEARS:
    records.append(result(year, "source", "yearly destinations exist", (POI_DIR / f"austria-{year}-pois.geoparquet").exists()))
    feature_dir = ROUTING_DATA / "features" / str(year)
    nearest_path = feature_dir / "nearest_infrastructure_100m.parquet"
    main_path = feature_dir / "accessibility_potentials_100m.parquet"
    long_path = feature_dir / "fachgruppe_accessibility_quarter_100m.parquet"
    firm_path = feature_dir / "firm_accessibility_quarter_100m.parquet"
    if nearest_path.exists():
        nearest = pd.read_parquet(nearest_path)
        records += [result(year, "nearest", "one row per active cell", len(nearest) == active_count), result(year, "nearest", "grid_id unique", nearest.grid_id.is_unique)]
    if main_path.exists():
        main = pd.read_parquet(main_path)
        required = {"grid_id", "year", "quarter", *main_access_columns()}
        records += [result(year, "grid-quarter", "required schema", required.issubset(main)), result(year, "grid-quarter", "year and four quarters complete", set(main.year) == {year} and set(main.quarter) == {1,2,3,4} and len(main) == active_count * 4), result(year, "grid-quarter", "no contemporaneous or Sparte fields", not any(c == "active_firms_t" or "sparte_" in c.lower() for c in main.columns))]
        records += numeric_checks(main, main_access_columns(), year, "grid-quarter")
    if long_path.exists():
        long = pd.read_parquet(long_path)
        required = {"grid_id", "year", "quarter", "Fachgruppe_ID", "same_fachgruppe_firms_access_15min", "same_fachgruppe_firms_access_30min"}
        coverage = long.groupby(["grid_id", "year", "quarter"])["Fachgruppe_ID"].nunique()
        records += [result(year, "Fachgruppe-long", "required schema", required.issubset(long)), result(year, "Fachgruppe-long", "all 95 Fachgruppen per grid-quarter", set(long.Fachgruppe_ID.astype(str)) == set(FACHGRUPPE_IDS) and (coverage == 95).all()), result(year, "Fachgruppe-long", "unique keys", not long.duplicated(["grid_id","year","quarter","Fachgruppe_ID"]).any())]
        records += numeric_checks(long, ["same_fachgruppe_firms_access_15min", "same_fachgruppe_firms_access_30min"], year, "Fachgruppe-long")
    if firm_path.exists() and long_path.exists() and main_path.exists():
        firm = pd.read_parquet(firm_path)
        required = {"firm_id", "grid_id_100m", "Fachgruppe_ID", "year", "quarter", "included_in_lagged_stock", "same_fachgruppe_firms_access_15min", "same_fachgruppe_firms_access_30min"}
        records.append(result(year, "firm-quarter", "required schema", required.issubset(firm)))
        firm["Fachgruppe_ID"] = firm["Fachgruppe_ID"].astype(str)
        long["Fachgruppe_ID"] = long["Fachgruppe_ID"].astype(str)
        probe = firm.merge(long, left_on=["grid_id_100m","year","quarter","Fachgruppe_ID"], right_on=["grid_id","year","quarter","Fachgruppe_ID"], suffixes=("", "_grid"))
        probe = probe.merge(main[["grid_id", "year", "quarter", "existing_firms_access_15min", "existing_firms_access_30min"]], left_on=["grid_id_100m", "year", "quarter"], right_on=["grid_id", "year", "quarter"], suffixes=("", "_total_grid"))
        exclusion = probe.included_in_lagged_stock.fillna(False).astype(int)
        for minutes in ACCESS_MINUTES:
            actual = probe[f"same_fachgruppe_firms_access_{minutes}min"]
            expected = (probe[f"same_fachgruppe_firms_access_{minutes}min_grid"] - exclusion).clip(lower=0)
            records.append(result(year, "firm-quarter", f"same-Fachgruppe {minutes}min matches long table with self-exclusion", np.allclose(actual, expected)))
            total_actual = probe[f"existing_firms_access_{minutes}min"]
            total_expected = (probe[f"existing_firms_access_{minutes}min_total_grid"] - exclusion).clip(lower=0)
            records.append(result(year, "firm-quarter", f"lagged-total {minutes}min matches grid table with self-exclusion", np.allclose(total_actual, total_expected)))
        records += numeric_checks(firm, ["same_fachgruppe_firms_access_15min", "same_fachgruppe_firms_access_30min"], year, "firm-quarter")

if STATUS_PATH.exists():
    status = pd.read_csv(STATUS_PATH)
    records.append(result("all", "manifest", "model products use lagged firm stock", "firm_mass_source" in status and set(status.firm_mass_source.dropna()) <= {"active_firms_tminus1"}))
if GRAPH_STATUS_PATH.exists():
    graphs = pd.read_csv(GRAPH_STATUS_PATH)
    graph_years = set(pd.to_numeric(graphs.year, errors="coerce").dropna().astype(int))
    manifest_rows = graphs.drop_duplicates("year", keep="last")
    records.append(result("all", "graphs", "successful graph manifests recorded for 2015-2025", graph_years >= set(YEARS) and (manifest_rows.status == "done").all() and manifest_rows.manifest_path_wsl.notna().all()))

validation = pd.DataFrame(records)
validation.to_csv(REPORT_PATH, index=False)
validation.groupby(["product", "status"]).size().rename("checks").reset_index()